In [1]:
!pip install llama-index
!pip install llama-index-embeddings-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 47.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.4/327.4 kB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.2/853.2 kB 6.8 MB/s eta 0:00:00
  Using cached nvidia_cuda_n

In [21]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
import os

In [23]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = None

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


LLM is explicitly disabled. Using MockLLM.


In [33]:
knowledge_base = [
    {
        "title": "Pythagorean Theorem",
        "description": "Illustrates the Pythagorean Theorem in a right-angled triangle.",
        "code": """
class PythagoreanTheorem(Scene):
    def construct(self):
        # Create the right-angled triangle
        triangle = Polygon(
            ORIGIN, RIGHT, RIGHT + UP,
            fill_color=BLUE, fill_opacity=0.5
        )
        square_a = Square().scale(3).next_to(triangle, LEFT, buff=0)
        square_b = Square().scale(4).next_to(triangle, DOWN, buff=0)
        square_c = Square().scale(5).next_to(triangle, UP + RIGHT, buff=0)
        self.play(Create(triangle))
        self.play(Create(square_a), Create(square_b), Create(square_c))
        self.wait(2)
        # Display the equation
        equation = MathTex('a^2 + b^2 = c^2').next_to(triangle, UP + RIGHT, buff=1)
        self.play(Write(equation))
        self.wait(2)
        """
    },
    {
        "title": "Quadratic Formula",
        "description": "Illustrates the quadratic formula and its components.",
        "code": """
class QuadraticFormula(Scene):
    def construct(self):
        # Display the quadratic formula
        formula = MathTex(
            'x = {-b \\pm \\sqrt{b^2-4ac} \\over 2a}'
        )
        self.play(Write(formula))
        self.wait(2)
        # Highlight each part
        discriminant = MathTex('\\sqrt{b^2-4ac}')
        self.play(Transform(formula[4:9], discriminant))
        self.wait(2)
        """
    }
]

contexts = [doc["description"] + ": " + doc["code"] for doc in knowledge_base]

In [34]:
output_dir = "test"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for i, context in enumerate(contexts):
  with open(os.path.join(output_dir, f"test{i}.txt"), "w") as f:
    f.write(context)

documents = SimpleDirectoryReader("test").load_data()

In [35]:
documents

[Document(id_='581bf561-5cb7-4915-8fff-4fd6e2cf87f3', embedding=None, metadata={'file_path': '/content/test/test0.txt', 'file_name': 'test0.txt', 'file_type': 'text/plain', 'file_size': 811, 'creation_date': '2024-06-24', 'last_modified_date': '2024-06-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text="Illustrates the Pythagorean Theorem in a right-angled triangle.: \nclass PythagoreanTheorem(Scene):\n    def construct(self):\n        # Create the right-angled triangle\n        triangle = Polygon(\n            ORIGIN, RIGHT, RIGHT + UP,\n            fill_color=BLUE, fill_opacity=0.5\n        )\n        square_a = Square().scale(3).next_to(triangle, LEFT, buff=0)\n        square_b = Square().scale(4).next_to(triangle, DOWN, buff=0)\n        square_c = 

In [36]:
index = VectorStoreIndex.from_documents(documents)
index

In [37]:
# set number of docs to retreive
top_k = 2

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=top_k,
)

In [38]:
query_engine = RetrieverQueryEngine(retriever=retriever, node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.5)])

In [39]:
query = "Generate a graph to illustrate the Pythagorean Theorem"
response = query_engine.query(query)
response.source_nodes[0].text

"Illustrates the Pythagorean Theorem in a right-angled triangle.: \nclass PythagoreanTheorem(Scene):\n    def construct(self):\n        # Create the right-angled triangle\n        triangle = Polygon(\n            ORIGIN, RIGHT, RIGHT + UP,\n            fill_color=BLUE, fill_opacity=0.5\n        )\n        square_a = Square().scale(3).next_to(triangle, LEFT, buff=0)\n        square_b = Square().scale(4).next_to(triangle, DOWN, buff=0)\n        square_c = Square().scale(5).next_to(triangle, UP + RIGHT, buff=0)\n        self.play(Create(triangle))\n        self.play(Create(square_a), Create(square_b), Create(square_c))\n        self.wait(2)\n        # Display the equation\n        equation = MathTex('a^2 + b^2 = c^2').next_to(triangle, UP + RIGHT, buff=1)\n        self.play(Write(equation))\n        self.wait(2)"

In [40]:
context = "Context:\n"
for i in range(top_k):
  context = context + response.source_nodes[i].text + "\n\n"

print(context)

Context:
Illustrates the Pythagorean Theorem in a right-angled triangle.: 
class PythagoreanTheorem(Scene):
    def construct(self):
        # Create the right-angled triangle
        triangle = Polygon(
            ORIGIN, RIGHT, RIGHT + UP,
            fill_color=BLUE, fill_opacity=0.5
        )
        square_a = Square().scale(3).next_to(triangle, LEFT, buff=0)
        square_b = Square().scale(4).next_to(triangle, DOWN, buff=0)
        square_c = Square().scale(5).next_to(triangle, UP + RIGHT, buff=0)
        self.play(Create(triangle))
        self.play(Create(square_a), Create(square_b), Create(square_c))
        self.wait(2)
        # Display the equation
        equation = MathTex('a^2 + b^2 = c^2').next_to(triangle, UP + RIGHT, buff=1)
        self.play(Write(equation))
        self.wait(2)

Illustrates the quadratic formula and its components.: 
class QuadraticFormula(Scene):
    def construct(self):
        # Display the quadratic formula
        formula = MathTex(
        

In [45]:
prompt_template = lambda user_query, context : f"""[INST]Manim Generator: functions as an illustrator of scientific (mathematical) concepts by generating manim code. \
It does not explain any concepts in response to the user query; all it outputs is the manim code for the relevant scientific concept asked by the user. Do not output anything else. \
Contexts and Examples: \n\n
{context} \n\n
Please respond to the following user query; use the above context if it is helpful: \  \n {query} \n [/INST] """


In [46]:
prompt_template(query, context)

"[INST]Manim Generator: functions as an illustrator of scientific (mathematical) concepts by generating manim code. It does not explain any concepts in response to the user query; all it outputs is the manim code for the relevant scientific concept asked by the user. Do not output anything else. Contexts and Examples: \n\n\nContext:\nIllustrates the Pythagorean Theorem in a right-angled triangle.: \nclass PythagoreanTheorem(Scene):\n    def construct(self):\n        # Create the right-angled triangle\n        triangle = Polygon(\n            ORIGIN, RIGHT, RIGHT + UP,\n            fill_color=BLUE, fill_opacity=0.5\n        )\n        square_a = Square().scale(3).next_to(triangle, LEFT, buff=0)\n        square_b = Square().scale(4).next_to(triangle, DOWN, buff=0)\n        square_c = Square().scale(5).next_to(triangle, UP + RIGHT, buff=0)\n        self.play(Create(triangle))\n        self.play(Create(square_a), Create(square_b), Create(square_c))\n        self.wait(2)\n        # Display 